## Proceso de limpieza en tabla de VENTAS

#### Importación de librerías y análisis exploratorio inicial

In [339]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata

In [476]:
dfventas_cp = pd.read_csv('./datos_crudos/ventas_empresa_dirty.csv')

In [477]:
dfventas_cp.shape

(150, 7)

In [478]:
dfventas_cp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_venta     150 non-null    int64  
 1   fecha_venta  104 non-null    object 
 2   producto     132 non-null    object 
 3   categoria    125 non-null    object 
 4   precio       131 non-null    object 
 5   cantidad     123 non-null    float64
 6   cliente      130 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 8.3+ KB


In [479]:
dfventas_cp.isna().sum()

id_venta        0
fecha_venta    46
producto       18
categoria      25
precio         19
cantidad       27
cliente        20
dtype: int64

In [480]:
dfventas_cp.apply(lambda col:col.duplicated().sum())

id_venta         2
fecha_venta     56
producto       141
categoria      144
precio         142
cantidad       145
cliente        142
dtype: int64

In [482]:
dfventas_cp.loc[dfventas_cp['id_venta'].duplicated(),'id_venta']

39    39
79    79
Name: id_venta, dtype: int64

In [483]:
dfventas_cp.sort_values

<bound method DataFrame.sort_values of      id_venta fecha_venta   producto   categoria   precio  cantidad  \
0           1  09/07/2023      Mouse  tecnología     1300       NaN   
1           2  2023-06-01   Notebook  Tecnologia     1100       2.0   
2           3         NaN        NaN  Accesorios     1100       5.0   
3           4  2023-07-14    TECLADO  TECNOLOGIA      800       2.0   
4           5  24/01/2023  notebook   TECNOLOGIA      NaN       2.0   
..        ...         ...        ...         ...      ...       ...   
145       146  21/07/2023      Mouse  tecnología     1100       1.0   
146       147  2023-07-28    Teclado  tecnología     1200       3.0   
147       148  2023-05-19  notebook          NaN     1200       2.0   
148       149  2023-07-22    Teclado  Tecnologia     1300       3.0   
149       150         NaN  notebook          NaN  1250usd       5.0   

        cliente  
0     empresa a  
1     Empresa A  
2    Cliente B   
3     Cliente B  
4    Cliente B   


#### Conclusión

La tabla analizada contiene 150 filas y 7 columnas. Se detectan valores nulos en la mayoría de las variables, destacándose la columna fecha_venta, con 46 valores faltantes. Dado que esta variable no resulta relevante para el análisis, se decide eliminarla. El resto de las columnas será conservado y depurado, corrigiendo errores de formato; en particular, la variable cantidad será convertida a tipo entero.

En cuanto a los duplicados, estos son esperables en las columnas producto, categoría, precio, cantidad y cliente. Sin embargo, los duplicados en id_venta serán analizados y corregidos, ya que se trata de un identificador único.

Asimismo, se detecta inconsistencia entre productos y precios, por lo que el proceso de limpieza comienza trabajando sobre la columna precio. A partir de esta, se corrige y completa la columna producto, utilizando valores estimativos coherentes con el mercado:

- Notebook: entre 1100 y 1300

- Monitor: 800

- Teclado: 45

- Mouse: 25

Estos valores se adoptan con fines analíticos, tomando como referencia la columna precio para asegurar consistencia en los datos.

#### Estandarización de formatos

In [484]:
cols_texto= dfventas_cp.select_dtypes(include='object').columns

dfventas_cp[cols_texto] = dfventas_cp[cols_texto].apply(
    lambda col: col.astype(str)
    .str.strip()
    .str.lower()
    .str.normalize('NFKD')
    .str.encode('ascii',errors='ignore')
    .str.decode('utf-8')
)

In [485]:
dfventas_cp[cols_texto] = dfventas_cp[cols_texto].replace('nan',pd.NA)

In [486]:
dfventas_cp

,id_venta,fecha_venta,producto,categoria,precio,cantidad,cliente
0,1,09/07/2023,mouse,tecnologia,1300,NaN,empresa a
1,2,2023-06-01,notebook,tecnologia,1100,2.0,empresa a
2,3,<NA>,<NA>,accesorios,1100,5.0,cliente b
3,4,2023-07-14,teclado,tecnologia,800,2.0,cliente b
4,5,24/01/2023,notebook,tecnologia,<NA>,2.0,cliente b
...,...,...,...,...,...,...,...
145,146,21/07/2023,mouse,tecnologia,1100,1.0,cliente b
146,147,2023-07-28,teclado,tecnologia,1200,3.0,empresa a
147,148,2023-05-19,notebook,<NA>,1200,2.0,empresa a
148,149,2023-07-22,teclado,tecnologia,1300,3.0,empresa a


#### Conclusión:
Se realiza la limpieza de variables de tipo string, eliminando espacios innecesarios y unificando los valores en mayúsculas para asegurar consistencia en los datos.

#### Conversión variable "Precio"

In [487]:
dfventas_cp['precio'] = (dfventas_cp['precio'].astype(str)
.str.lower()
.str.replace("usd","",regex=False)
.str.replace("$","",regex=False)
.str.strip())

In [488]:
dfventas_cp['precio'] = pd.to_numeric(dfventas_cp['precio'],errors='coerce')

In [489]:
dfventas_cp['precio'].unique()

array([1300., 1100.,  800.,   nan, 1250.,   25.,   45., 1200.])

#### Tarea realizada en conversión de columna 'precio'
Se procede a eliminación de simbolos y letras en la columna 'precio', y conversion del tipo de la misma

#### Tratamiento columna 'producto'

In [490]:
dfventas_cp.loc[dfventas_cp['precio'].between(1100,1300),'producto'] = 'notebook'

In [491]:
dfventas_cp.loc[dfventas_cp['precio'] == 800,'producto'] = 'monitor'

In [492]:
dfventas_cp.loc[dfventas_cp['precio'] == 45,'producto'] = 'teclado'

In [493]:
dfventas_cp.loc[dfventas_cp['precio'] == 25,'producto'] = 'mouse'

In [494]:
dfventas_cp.loc[(dfventas_cp['producto'].isnull()) &
    (dfventas_cp['precio'].isnull()) &
        (dfventas_cp['categoria'] == 'tecnologia'),'producto'] = 'notebook'

In [495]:
dfventas_cp = dfventas_cp[dfventas_cp['id_venta'] != 123]

In [496]:
dfventas_cp.loc[dfventas_cp['id_venta'] ==118,'producto'] = 'no_informado'

#### Tareas realizadas en columna 'producto'
Se realiza la conversión de los valores de la variable producto en función del precio, aplicando los criterios definidos previamente.
Los precios comprendidos entre 1100 y 1300 se asignan al producto notebook, el valor 800 a monitor, 45 a teclado y 25 a mouse.

Luego de esta corrección, permanecen 4 registros con valores nulos que no cuentan con información suficiente de precio.
En dos de estos casos, la categoría figura como tecnología, por lo que se deduce que el producto corresponde a notebook, al ser el único producto asociado a dicha categoría. A partir de esta inferencia, también es posible estimar el precio correspondiente.

En otro registro, no se dispone de información relevante en las variables cliente, cantidad, precio ni producto, por lo que se decide eliminarlo del dataset.

Por último, en el registro restante, no es posible deducir el producto, ya que la categoría es accesorios, la cual incluye múltiples productos posibles. Si bien no se cuenta con el precio, sí se dispone de información sobre cliente y cantidad, variables útiles para otros análisis. Por este motivo, se decide asignar el valor "no_informado" a la variable producto.

#### Tratamiento columna 'categoria'

In [498]:
dfventas_cp.loc[dfventas_cp['producto']=='mouse','categoria'] = 'accesorios'

In [499]:
dfventas_cp.loc[dfventas_cp['producto'] == 'notebook','categoria'] = 'tecnologia'

In [500]:
dfventas_cp.loc[dfventas_cp['producto']=='teclado','categoria'] ='accesorios'

In [501]:
dfventas_cp.loc[dfventas_cp['producto'] == 'monitor','categoria'] = 'accesorios'

#### Tareas realizadas en columna 'categoria'
Respecto al tratamiento de valores nulos, se priorizó la recuperación de registros por sobre su eliminación.
En la columna categoria, los valores nulos y errores se corrigieron utilizando la información disponible en la columna producto y criterios de negocio, dado que no se cuenta con documentación oficial de la empresa. De esta forma, se clasificó mouse y teclado como accesorios, y notebook como tecnología.
Se identificaron dos registros restantes con valores nulos tanto en categoria como en producto. Uno de ellos fue eliminado debido a la falta adicional de información del cliente, mientras que el otro se conservó y se completó con el valor “no_informado”, ya que, si bien carece de información esencial, posee datos válidos necesarios para los análisis de volumen y comportamiento de compra.

#### Se realiza copia del proceso de limpieza

In [577]:
df_limpio = dfventas_cp.copy()

In [589]:
df_limpio2 = df_limpio.copy()

#### Tratamiento columna 'precio'

In [578]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'notebook') &
        (df_limpio['cliente'] == 'cliente b'),'precio'
] = 1250

In [579]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'mouse') &
        (df_limpio['cliente'] == 'cliente b'),'precio'
] = 25

In [580]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'teclado') &
        (df_limpio['cliente'] == 'cliente b'),'precio'] = 45


In [581]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'mouse') &
        (df_limpio['cliente'] == 'cliente c'),'precio'
] = 25

In [582]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] =='notebook') &
        (df_limpio['cliente'] == 'cliente c'),'precio'
] = 1250

In [583]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'mouse') &
        (df_limpio['cliente']=='empresa a'),'precio'
] = 25

In [584]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'notebook') &
        (df_limpio['cliente'] == 'empresa a'),'precio'
] = 1250

In [585]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] == 'teclado') &
        (df_limpio['cliente'] == 'empresa a'),'precio'
] = 45

In [586]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['producto'] =='teclado') &
        (df_limpio['cliente'] =='empresa d'),'precio'
] = 45

In [587]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['cliente'].isnull()) &
        (df_limpio['producto']=='monitor'),'precio'
] = 800

In [588]:
df_limpio.loc[(df_limpio['precio'].isnull()) &
    (df_limpio['cliente'].isnull()) &
        (df_limpio['producto'] == 'notebook'),'precio'
] = 1250

#### Tareas realizadas en columna 'precio'

Para los registros con precio faltante, se imputaron valores únicamente cuando el producto estaba identificado. En el caso de notebooks, se utilizó un valor representativo de 1250, correspondiente al valor central del rango observado (1100–1300).

Cuando la cantidad no estaba informada, se asignó el valor mínimo de 1, asumiendo una venta individual.

En los casos donde no fue posible inferir ni el producto ni el precio a partir de información contextual (cliente, fecha o categoría), los valores se mantuvieron como nulos o se clasificaron como no_informado, evitando introducir supuestos no sustentados por los datos.

#### Tratamiento columna 'cantidad'

In [596]:
df_limpio2.loc[df_limpio2['cantidad'].isnull(),'cantidad'] = 1

In [614]:
df_limpio2['cantidad'] = df_limpio2['cantidad'].astype(int)

####  Tareas realizadas en columna 'cantidad'
Para los registros con cantidad faltante, se imputó el valor mínimo de 1 unidad, asumiendo que la existencia de la venta implica al menos una unidad vendida. Esta decisión se tomó al no contar con información suficiente para estimar una cantidad superior sin introducir supuestos no sustentados por los datos. Se realiza conversión de la misma a tipo entero

#### Tratamiento columna 'cliente'

In [609]:
df_limpio2['cliente'] = df_limpio2['cliente'].fillna('cliente_no_informado')

#### Tareas realizadas en columna 'cliente'
En el proceso de limpieza de datos se analizaron los registros con valores faltantes en la variable cliente. Se intentó inferir esta información utilizando variables contextuales como la fecha de venta, el tipo de producto y el comportamiento histórico de compra.

Sin embargo, tras realizar la verificación correspondiente, no se identificaron patrones consistentes ni relaciones que permitieran deducir de manera confiable el cliente asociado a dichas transacciones. Dado que asignar un cliente de forma arbitraria implicaría introducir información incorrecta en el dataset, se optó por no realizar imputaciones basadas en supuestos no verificables.

En consecuencia, los valores faltantes fueron tratados como cliente_no_informado, preservando la integridad de los datos y evitando sesgos en análisis posteriores.

#### Tratamiento columna 'fecha_venta'

In [612]:
df_limpio2 = df_limpio2.drop('fecha_venta',axis=1)

#### Tareas realizadas en columna 'fecha_venta'
Se procede a eliminación de la misma ya que es innecesaria para nuestro análisis

#### Se realiza guardado final de archivo

In [622]:
df_limpio2.to_csv('ventas_limpio.csv',index=False)

In [628]:
df_limpio2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149 entries, 0 to 149
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id_venta   149 non-null    int64  
 1   producto   149 non-null    object 
 2   categoria  149 non-null    object 
 3   precio     148 non-null    float64
 4   cantidad   149 non-null    int64  
 5   cliente    149 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 8.1+ KB
